# 02 — Assessment Ratio Map

Choropleth map of California counties colored by **assessment ratio** (assessed value / market value).

- **Lower ratio** → assessed value is far below market value → **greater Prop 13 benefit** (long-tenure owners)
- **Higher ratio** → assessed value close to market value → recent purchases or new construction

Under Prop 13, properties are reassessed to market value only upon sale. Long-held homes retain 1978-era assessments grown at ≤2%/year, creating large gaps in high-appreciation areas.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'county_prop13.csv')
df['fips'] = df['fips'].astype(str).str.zfill(5)
print(f'Loaded {len(df)} counties')

Loaded 58 counties


In [2]:
# Choropleth: assessment ratio by county
fig = px.choropleth(
    df,
    geojson='https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json',
    locations='fips',
    color='assessment_ratio',
    color_continuous_scale='RdYlGn',
    range_color=[0.3, 1.0],
    scope='usa',
    hover_name='county',
    hover_data={
        'assessment_ratio': ':.1%',
        'tax_gap_per_household': ':$,.0f',
        'fips': False,
    },
    labels={
        'assessment_ratio': 'Assessment Ratio',
        'tax_gap_per_household': 'Annual Tax Gap/HH',
    },
    title='Prop 13 Assessment Ratio by CA County<br><sup>Assessed Value ÷ Market Value — lower = more Prop 13 benefit</sup>',
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(
    margin={'l': 0, 'r': 0, 't': 60, 'b': 0},
    coloraxis_colorbar=dict(
        title='Assessment<br>Ratio',
        tickformat='.0%',
    ),
    height=600,
)
fig.show()

In [3]:
# Summary stats
low = df.nsmallest(5, 'assessment_ratio')[['county','assessment_ratio','tax_gap_per_household']]
high = df.nlargest(5, 'assessment_ratio')[['county','assessment_ratio','tax_gap_per_household']]
print('Most Prop 13 benefit (lowest assessment ratio):')
print(low.to_string(index=False))
print()
print('Least Prop 13 benefit (highest assessment ratio):')
print(high.to_string(index=False))

Most Prop 13 benefit (lowest assessment ratio):
       county  assessment_ratio  tax_gap_per_household
      Trinity          0.293887            4655.851661
    San Mateo          0.528350            9628.376817
Santa Barbara          0.534185            5593.239304
     Tuolumne          0.547736            2447.944235
       Amador          0.554733            2474.857918

Least Prop 13 benefit (highest assessment ratio):
    county  assessment_ratio  tax_gap_per_household
      Kern          0.865652             481.381596
 Riverside          0.810855            1066.257949
  Imperial          0.776312             709.376842
    Placer          0.754382            1875.021863
San Benito          0.723736            2351.980599
